In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const
import utilities.plot_settings
from matplotlib.collections import PathCollection
from matplotlib.legend_handler import HandlerPathCollection, HandlerLine2D







def update(handle, orig):
    handle.update_from(orig)
    handle.set_alpha(1)
    handle.set_markersize(3)


In [ ]:
def B_from_timing(P: float, Pdot: float) -> float:
    """
    B field estimated from timing properties. 
    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        Pdot (float): period derivative of a simulated pulsar in [s/s].   

    Returns:
        (float): value of the dipolar component of the magnetic field at the
        magnetic pole for a simulated neutron star, measured in [G].
    """

    # Period derivative.
    B = np.sqrt(P * Pdot / (4 * np.pi**2 * beta_1))

    return B

def Edot_from_timing(P: float, Pdot: float) -> float:
    """
    Pulsar rotational power loss from timing properties. 
    
    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        Pdot (float): period derivative of a simulated pulsar in [s/s].   

    Returns:
        (float): characteristic age in [yr].     
    """

    # Period derivative.
    Erot_dot = (2.*np.pi)**2 * NS_inertia * Pdot / P**3

    return Erot_dot




# Characteristic neutron star radius in [cm].
NS_radius = cfg["NS_radius"] #7.e8
# Characteristic neutron star mass in solar masses.
NS_mass = cfg["NS_mass"]
# Dimensionless coefficients k_0, k_1, k_2 for a force-free magnetosphere
# taken from Spitkovsky (2006) and Philippov et al. (2014).
# For comparison, in vacuum k_0 = 0 and k_1 = k_2 = 2/3.
k_coefficients = [1.0, 1.0, 1.0]
# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * NS_mass * NS_radius ** 2
# Auxiliary quantity beta as defined in eq. (72) of Pons & Vigano (2019).
beta = 1./4. * NS_radius ** 6 / (NS_inertia * const.C ** 3)
#beta = np.pi ** 2 * NS_radius ** 6 / (NS_inertia * const.c ** 3)
print(beta)
# Assume an inclination angle in [rad].
chi = 0.
beta_1 = beta * (k_coefficients[0] + k_coefficients[1] * np.sin(chi) ** 2)


In [ ]:
data_i_2 = pd.read_pickle(
    "/home/celsa/Documents/MAGNESIA_population_synthesis/toremove/nanda_experiment/NS_exp_2_B2comp/initial_population.pkl.gz",
    compression="gzip",
)

data_f_2 = pd.read_pickle(
    "/home/celsa/Documents/MAGNESIA_population_synthesis/toremove/nanda_experiment/NS_exp_2_B2comp/final_population.pkl.gz",
    compression="gzip",
)


In [ ]:
data_i_4 = pd.read_pickle(
    "/home/celsa/Documents/MAGNESIA_population_synthesis/toremove/nanda_experiment/NS_exp_4_B2comp/initial_population.pkl.gz",
    compression="gzip",
)

data_f_4 = pd.read_pickle(
    "/home/celsa/Documents/MAGNESIA_population_synthesis/toremove/nanda_experiment/NS_exp_4_B2comp/final_population.pkl.gz",
    compression="gzip",
)


In [ ]:
P_i_2 = data_i_2["P"]["[s]"].to_numpy()
P_dot_i_2 = data_i_2["P_dot"]["[s s^-1]"].to_numpy()
B_i_2 = data_i_2["B"]["[G]"].to_numpy()


P_f_2 = data_f_2["P"]["[s]"].to_numpy()
P_dot_f_2 = data_f_2["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol_2 = data_f_2["L_radio_bol"]["[erg s^-1]"].to_numpy()
B_f_2 = data_f_2["B"]["[G]"].to_numpy()

m = cfg["NS_mass"]
R = cfg["NS_radius"]
I = (2/5)*m*R**2
Erot_f_2 = ((2*np.pi)**2)*I*(P_dot_f_2/P_f_2**3)

intercept_radio_2 = data_f_2[('intercepted_radio',' ')] == 1
data_f_2[intercept_radio_2].head()

In [ ]:
P_i_4 = data_i_4["P"]["[s]"].to_numpy()
P_dot_i_4 = data_i_4["P_dot"]["[s s^-1]"].to_numpy()
B_i_4 = data_i_4["B"]["[G]"].to_numpy()


P_f_4 = data_f_4["P"]["[s]"].to_numpy()
P_dot_f_4 = data_f_4["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol_4 = data_f_4["L_radio_bol"]["[erg s^-1]"].to_numpy()
B_f_4 = data_f_4["B"]["[G]"].to_numpy()

m = cfg["NS_mass"]
R = cfg["NS_radius"]
I = (2/5)*m*R**2
Erot_f_4 = ((2*np.pi)**2)*I*(P_dot_f_4/P_f_4**3)

intercept_radio_4 = data_f_4[('intercepted_radio',' ')] == 1

In [ ]:
from matplotlib.ticker import ScalarFormatter

P_bins = np.logspace(-2, 6, 31)

figsize=(12, 10)
# Create the figure and axes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={"height_ratios": [1, 3], "hspace": 0})



# -----------------------Plot the distribution of P_i_1 at the top-----------------

ax1.hist(P_i_2, bins=P_bins,
    histtype="step",
    edgecolor="grey",
    lw=3,)
ax1.hist(P_i_4, bins=P_bins,
    histtype="step",
    edgecolor="lightgrey",
    lw=3,)


ax1.hist(P_f_2[intercept_radio_2],
     bins=P_bins,
     histtype="step",
     edgecolor="royalblue",
         lw=3,
        )

ax1.hist(P_f_2,
     bins=P_bins,
     histtype="step",
     edgecolor="lightskyblue",
         lw=3,
         ls= '--'
        )

ax1.hist(P_f_4[intercept_radio_4],
     bins=P_bins,
     histtype="step",
     edgecolor="tab:pink",
         lw=3,
        )

ax1.hist(P_f_4,
     bins=P_bins,
     histtype="step",
     edgecolor="lightpink",
         lw=3,
         ls= '--'
        )
#ax1.text(1e4,1e5,fontsize = 25)

ax1.set_yscale('log')
ax1.set_yticks([1e2, 1e4, 1e6])
ax1.set_ylabel("# NSs")
ax1.tick_params(axis="x", labelbottom=False)
#ax1.legend(markerscale=0,loc='upper right')
plt.xscale('log') 
plt.yscale('log') 

#------------------------Plot the B and Edot lines--------------------------------
P_min = 1e-3
P_max = 1e7
Pdot_min = 1e-28
Pdot_max = 1e-2

log_P_edges = np.linspace(np.log10(P_min), np.log10(P_max), 71)
log_P_centers = 0.5 * (log_P_edges[1:] + log_P_edges[:-1])
P_edges = 10**log_P_edges
P_centers = 10**log_P_centers

log_Pdot_edges = np.linspace(np.log10(Pdot_min), np.log10(Pdot_max), 71)
log_Pdot_centers = 0.5 * (log_Pdot_edges[1:] + log_Pdot_edges[:-1])
Pdot_edges = 10**log_Pdot_edges
Pdot_centers = 10**log_Pdot_centers

P_grid, Pdot_grid = np.meshgrid(P_centers, Pdot_centers, indexing='ij')
B_timing = B_from_timing(P_grid, Pdot_grid)

contour_B = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(B_timing), 
    levels = np.array([6.,9.,12.,15.,18]), 
    colors='black',
    linestyles='dashed',
    alpha = 0.7,
    #interpolation='none'
)

fmt = {}
strs = ['$10^{6}$ G', '$10^{9}$ G', '$10^{12}$ G', '$10^{15}$ G','$10^{17}$ G']
for l,s in zip( contour_B.levels, strs ):
    fmt[l] = rf"{s}"

manual_locations = [(8e-3, 1e-26), (1e5, 1e-25), (5e-3, 1e-12), (4e-3, 1e-6), (1e0, 1e-4)]

ax2.clabel(
    contour_B, 
    contour_B.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20, 
    colors='black'
)


#-------------------------------------Edot constant lines-----------------------------------------
Edot_timing = Edot_from_timing(P_grid, Pdot_grid)

contour_Edot = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(Edot_timing),
    levels = np.array([12.,20.,29.,38.,47.]), 
    colors='black',
    linestyles='dashed',
    alpha = 0.7,

    #interpolation='none'
)

fmt = {}
strs = ['$10^{12}$ erg s$^{-1}$', '$10^{20}$ erg s$^{-1}$', '$10^{29}$ erg s$^{-1}$', '$10^{38}$ erg s$^{-1}$', '$10^{47}$ erg s$^{-1}$']

for l,s in zip( contour_Edot.levels, strs ):
    fmt[l] = rf"{s}"
    
manual_locations = [(1e6, 1e-20), (1e6, 1e-12), (1e3, 1e-8), (1e1, 1e-5), (9e-2, 1e-6)]

ax2.clabel(
    contour_Edot, 
    contour_Edot.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20,
    colors='black'
)



#----------------------------Plot of the points in the 2D-------------------------------------
ax2.plot(
    P_i_4,
    P_dot_i_4,
    linestyle="None",
    marker="o",
    color="lightgrey",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Initial population NS4$^{*}$_Bconst'
)

ax2.plot(
    P_i_2,
    P_dot_i_2,
    linestyle="None",
    marker="o",
    color="grey",
    markersize=2,
    alpha=0.2,
    rasterized=True,
    label = 'Initial population NS2$^{*}$_Bconst'
)

ax2.plot(
    P_f_4,
    P_dot_f_4,
    linestyle="None",
    marker="o",
    color="lightpink",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Final population NS4$^{*}$_Bconst'
)



ax2.plot(
    P_f_4[intercept_radio_4],
    P_dot_f_4[intercept_radio_4],
    linestyle="None",
    marker="o",
    color="tab:pink",
    markersize=0.5,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los NS4$^{*}$_Bconst'
)


ax2.plot(
    P_f_2,
    P_dot_f_2,
    linestyle="None",
    marker="o",
    color="lightskyblue",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Final population NS2$^{*}$_Bconst'
)



ax2.plot(
    P_f_2[intercept_radio_2],
    P_dot_f_2[intercept_radio_2],
    linestyle="None",
    marker="o",
    color="royalblue",
    markersize=0.5,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los NS2$^{*}$_Bconst'
)


# Get the handles and labels of the plotted lines
handles, labels = plt.gca().get_legend_handles_labels()

# Create two separate legend blocks
legend1 = plt.legend(handles[:2], labels[:2], loc='upper right', frameon=False,  fontsize=20,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})
legend2 = plt.legend(handles[2:], labels[2:], loc='lower left', frameon=False, fontsize=20,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})

# Add the first legend to the plot
plt.gca().add_artist(legend1)


ax2.set_xlabel(r"$P$ [s]")
ax2.set_ylabel(r"$\dot{P}$ [s s$^{-1}$]")
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim(1e-3,1e7)
ax2.set_ylim(1e-28,1e-3)
#plt.legend(frameon=False, loc='lower left', fontsize=20,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),plt.Line2D : HandlerLine2D(update_func = update)})

# Remove the space between the subplots
plt.subplots_adjust(hspace=0)
plt.savefig('/home/celsa/Documents/paper_nandaetal_2023/figure3_top_right.png',format= 'png')

# Show the plot
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.plot(
    P_f_4,
    Erot_f_4,
    linestyle="None",
    marker="o",
    color="lightpink",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Final population NS4$^{*}$_Bconst'
)

ax.plot(
    P_f_4[intercept_radio_4],
    Erot_f_4[intercept_radio_4],
    linestyle="None",
    marker="o",
    color="tab:pink",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los NS4$^{*}$_Bconst'
)

ax.plot(
    P_f_2,
    Erot_f_2,
    linestyle="None",
    marker="o",
    color="lightskyblue",
    markersize=3,
    alpha=0.5,
    rasterized=True,
    label = 'Final population NS2$^{*}$_Bconst'
)

ax.plot(
    P_f_2[intercept_radio_2],
    Erot_f_2[intercept_radio_2],
    linestyle="None",
    marker="o",
    color="royalblue",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Intercept our los NS2$^{*}$_Bconst'
)









#plt.axvline(10**(-0.7))
ax.set_ylabel(r"$\dot{E} \, [\rm erg \, s^{-1}]$")
ax.set_xlabel(r"$P$ [s]")
plt.xscale('log') 
plt.yscale('log') 
plt.xlim(1e-3,1e7)
plt.ylim(1e5,1e40)
#plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20,markerscale=5)
plt.legend(frameon=False, loc=0, fontsize=25,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})
plt.grid()

#plt.savefig()
plt.savefig('/home/celsa/Documents/paper_nandaetal_2023/figure1_top_right.png',format= 'png')

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Create a plot
plt.plot([1, 2, 3], label='Data 1')
plt.plot([4, 5, 6], label='Data 2')
plt.plot([7, 8, 9], label='Data 3')

# Get the handles and labels of the plotted lines
handles, labels = plt.gca().get_legend_handles_labels()

# Create two separate legend blocks
legend1 = plt.legend(handles[:1], labels[:1], loc='upper right', title='Legend 1')
legend2 = plt.legend(handles[1:], labels[1:], loc='lower left', title='Legend 2')

# Add the first legend to the plot
plt.gca().add_artist(legend1)

# Show the plot
plt.show()
